## Principles of Machine Learning Final Project
**By: Saheli Ray, Wyatt Golden, HuiDi Hu**

In [2]:
# All imports needed for the project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import warnings
warnings.filterwarnings("ignore")

pd.options.display.max_columns = 100

In [6]:
# !! if you PWD is not /home/jovyan you are inside the wrong environment and cannot guarantee proper behavior !!
! pwd
! ls work

/home/jovyan
FinalProject.ipynb  sample_submission.csv  train_clean.csv  train_trimmed.csv
README.md	    test.csv		   train.csv


## Data Exploration / Cleaning
**In this section of the notebook, we will explore the dataset, seeing what it contains and figure out how we will modify it in order to make it the best for the model**

In [14]:
# Load dataset and look at it's shape and first 5 rows
train = pd.read_csv("work/train.csv", low_memory=False)
print("Dataset shape:", train.shape)
train.head(5)

Dataset shape: (307178, 55)


,INDEX_NR,INCIDENT_DATE,INCIDENT_MONTH,INCIDENT_YEAR,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,LATITUDE,LONGITUDE,RUNWAY,STATE,FAAREGION,LOCATION,OPID,OPERATOR,REG,FLT,AIRCRAFT,AMA,AMO,EMA,EMO,AC_CLASS,AC_MASS,TYPE_ENG,NUM_ENGS,ENG_1_POS,ENG_2_POS,ENG_3_POS,ENG_4_POS,PHASE_OF_FLIGHT,HEIGHT,SPEED,DISTANCE,SKY,PRECIPITATION,BIRD_BAND_NUMBER,SPECIES_ID,SPECIES,OUT_OF_RANGE_SPECIES,REMARKS,REMAINS_COLLECTED,REMAINS_SENT,WARNED,NUM_SEEN,NUM_STRUCK,SIZE,ENROUTE_STATE,COMMENTS,SOURCE,PERSON,LUPDATE,TRANSFER,INDICATED_DAMAGE
0,1410120,12/13/93,12,1993,NaN,Day,TJSJ,LUIS MUNOZ MARIN INTL,18.43942,-66.00183,7,PR,ASO,NaN,AAL,AMERICAN AIRLINES,N892AA,NaN,B-727-200,148,11,34.0,10.0,A,4.0,D,3.0,5.0,6.0,5.0,NaN,Approach,300.0,145.0,NaN,Some Cloud,NaN,NaN,UNKBS,Unknown bird - small,0,NO SIGN OF BIRD ON A/C.,1,0,No,10-Feb,10-Feb,Small,NaN,NaN,FAA Form 5200-7,Pilot,4/3/23,0,0
1,709688,2/1/10,2,2010,5:00,Night,WMKK,KUALA LUMPUR INTL,2.745578,101.709917,32R,FN,FGN,NaN,FDX,FEDEX EXPRESS,N608FE,5293,MD-11,583,39,22.0,7.0,A,4.0,D,3.0,1.0,6.0,1.0,NaN,Approach,50.0,NaN,0.0,NaN,NaN,NaN,UNKBM,Unknown bird - medium,0,EVID OF STRIKE FOUND ON LOWER RT SIDE OF RADOME.,0,0,Unknown,NaN,1,Medium,NaN,2010-5-18-53374 /Legacy Record 300758/,FAA Form 5200-7-E,Air Transport Operations,6/9/10,0,0
2,730841,5/9/12,5,2012,2:00,Night,KSDF,MUHAMMAD ALI INTERNATIONAL,38.17439,-85.736,35L,KY,ASO,NaN,UPS,UPS AIRLINES,N141UP,907,A-300,04A,1,34.0,46.0,A,4.0,D,2.0,1.0,1.0,NaN,NaN,Approach,3500.0,240.0,8.0,NaN,NaN,NaN,UNKBL,Unknown bird - large,0,"STARTED TO SLOW DOWN FROM 250 KTS AT AROUND 4,...",0,0,No,NaN,1,Large,NaN,UPS EVENT REPT 36216 (4/22/13 UPDATED COST) /L...,Air Transport Report,Air Transport Operations,4/22/13,0,1
3,654676,10/8/02,10,2002,NaN,NaN,KLAX,LOS ANGELES INTL,33.94254,-118.40807,25R,CA,AWP,NaN,UNK,UNKNOWN,NaN,NaN,UNKNOWN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NE120,Western gull,0,REMAINS OF 2 GULLS WERE PICKED UP OF RWY.,1,0,Unknown,NaN,10-Feb,Medium,NaN,2002-10-8-111929 /Legacy Record 216397/,FAA Form 5200-7-E,Carcass Found,1/9/03,0,0
4,629708,2/3/97,2,1997,NaN,Dawn,PHLI,LIHUE ARPT,21.97598,-159.33896,35,HI,AWP,NaN,1AAH,ALOHA AIRLINES,NaN,NaN,B-737-200,148,13,34.0,10.0,A,4.0,D,2.0,1.0,1.0,NaN,NaN,Landing Roll,0.0,135.0,0.0,Some Cloud,NaN,NaN,R1101,American barn owl,0,TIME 0824 LCL.,0,0,No,1,1,Medium,NaN,SOURCE 5200-7 & PACIR /Legacy Record 121531/,Multiple,NaN,3/1/07,0,0


## Selected Features for Model Training

This dataset contains many variables, but for this first modeling pass we chose fields that are both predictive and reliable. The target variable `INDICATED_DAMAGE` is also included.

### Kept for the model
- `LATITUDE` / `LONGITUDE`: geographic position gives useful location signal without relying on high-cardinality airport codes.
- `AC_CLASS`, `AC_MASS`, `TYPE_ENG`: aircraft size, class, and engine type are good proxies for vulnerability and damage potential.
- `SPECIES`: the type of wildlife is a strong indicator of the likely impact.
- `SIZE`: animal size is directly related to damage severity.
- `SPEED`: aircraft speed at impact is important for energy transfer and damage likelihood.
- `NUM_STRUCK`: number of animals struck could correlate with damage extent.
- `INDICATED_DAMAGE`: target variable indicating whether damage occurred.

### Dropped by design for now
- `PHASE_OF_FLIGHT`: we decided not to use this for prediction, even though it could influence damage, because it may add noise without a strong signal for classification.
- `HEIGHT`: also omitted to keep the model focused on simpler, more direct predictors.
- `SKY`: weather/sky condition is less reliable and not included in this pass.
- `INCIDENT_MONTH`, `TIME_OF_DAY`: temporal factors may not directly affect damage severity post-strike.

### Other variables not used for this first model
- Record identifiers and timestamps: `INDEX_NR`, `INCIDENT_DATE`, `INCIDENT_YEAR`, `LUPDATE`, `TRANSFER`.
- Airport/location identifiers: `AIRPORT_ID`, `AIRPORT`, `STATE`, `FAAREGION`, `LOCATION`, `ENROUTE_STATE`.
- Operator/aircraft metadata: `OPID`, `OPERATOR`, `REG`, `FLT`, `AIRCRAFT`, `AMA`, `AMO`, `EMA`, `EMO`.
- Engine configuration details: `NUM_ENGS`, `ENG_1_POS`, `ENG_2_POS`, `ENG_3_POS`, `ENG_4_POS`.

Most of these fields are excluded because they are either high-cardinality, too sparse, redundant with stronger features, or more appropriate for later experimentation once the baseline model is stable.

- Other strike context: `RUNWAY`, `DISTANCE`, `PRECIPITATION`, `BIRD_BAND_NUMBER`, `SPECIES_ID`, `OUT_OF_RANGE_SPECIES`, `REMARKS`, `REMAINS_COLLECTED`, `REMAINS_SENT`, `WARNED`, `NUM_SEEN`, `SOURCE`, `PERSON`.

In [15]:
# The feature values we will be using to train our model, plus the target.
selected_columns = [
    "LATITUDE",
    "LONGITUDE",
    "AC_CLASS",
    "AC_MASS",
    "TYPE_ENG",
    "SPEED",
    "SPECIES",
    "SIZE",
    "NUM_STRUCK",
    "INDICATED_DAMAGE",  # Target variable
]

train_trimmed = train[selected_columns].copy()

# Take a look at the trimmed dataset's shape and first 5 rows
print(train_trimmed.shape)
print(train_trimmed.head())

# Save this trimmed dataset to a new CSV file for easier access in the future
train_trimmed.to_csv("work/train_trimmed.csv", index=False)

(307178, 10)
   LATITUDE   LONGITUDE AC_CLASS  AC_MASS TYPE_ENG  SPEED  \
0  18.43942   -66.00183      A        4.0        D  145.0   
1  2.745578  101.709917      A        4.0        D    NaN   
2  38.17439     -85.736      A        4.0        D  240.0   
3  33.94254  -118.40807      NaN      NaN      NaN    NaN   
4  21.97598  -159.33896      A        4.0        D  135.0   

                 SPECIES    SIZE NUM_STRUCK  INDICATED_DAMAGE  
0   Unknown bird - small   Small     10-Feb                 0  
1  Unknown bird - medium  Medium          1                 0  
2   Unknown bird - large   Large          1                 1  
3           Western gull  Medium     10-Feb                 0  
4      American barn owl  Medium          1                 0  


## Target Variable Distribution Comparison

Before dropping rows with missing values, let's compare the target variable distribution between the full dataset and the subset that will remain after cleaning. This ensures we're not introducing bias by dropping missing data.

In [16]:
print("Target distribution before dropping rows with missing values in any feature:")
print(train_trimmed['INDICATED_DAMAGE'].value_counts(normalize=True) * 100)
print()

# Drop rows with missing values in all selected features
features_to_check = [col for col in selected_columns]
train_clean = train_trimmed.dropna(subset=features_to_check)

print("Target distribution after dropping rows with missing values in any feature:")
print(train_clean['INDICATED_DAMAGE'].value_counts(normalize=True) * 100)
print()

print(f"Rows before dropping missing features: {len(train_trimmed)}")
print(f"Rows after dropping missing features: {len(train_clean)}")
print(f"Rows dropped: {len(train_trimmed) - len(train_clean)}")

# Save this trimmed dataset to a new CSV file for easier access in the future
train_clean.to_csv("work/train_clean.csv", index=False)

Target distribution before dropping rows with missing values in any feature:
INDICATED_DAMAGE
0    93.643099
1     6.356901
Name: proportion, dtype: float64

Target distribution after dropping rows with missing values in any feature:
INDICATED_DAMAGE
0    89.102099
1    10.897901
Name: proportion, dtype: float64

Rows before dropping missing features: 307178
Rows after dropping missing features: 84163
Rows dropped: 223015


**Since we still have a lot of records to train on and the class distriution actually seemed to improve, we think it is safe to continue with just dropping the missing values**